# XGBoost Regressor using wastewater to predict ed visits


1. Loading packages and data.
2. Processing data: filter features, labels and dates.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os 
sys.path.append('..')
from TARnISHED-WW.helper_functions import compute_mae_test, create_lagged_features_by_region
from TARnISHED-WW.training import rolling_window_xgboost, train_and_predict_per_region_xgboost, train_and_predict_per_region_xgboost_bootstrapping
from TARnISHED-WW.plots_function import plot_train_test_predictions,plot_train_test_predictions_bootstrapping,resampling_weekly

import numpy as np

suffix = ""

features =['total_cases_covid','load_trillion_covid','total_cases_flua','load_trillion_flua','total_cases_rsv','load_trillion_rsv',"tests_per_capita"
           ]#
label = ['total_ed_visits']

#Load dataset
root_dir = Path(os.path.dirname(Path.cwd()))
filepath = os.path.join(root_dir, "data", "processed")
prediction_length = 21 #days
freq = "D"
pop_folder = Path("O:/BCCDC/Groups/Analytics/Projects/covid_modeling/06 Projects/RSV Flu Modelling/Data/wwtp_postal_codes")

pop = pd.read_csv(os.path.join(pop_folder, "population_of_wwtp.csv"))
df_raw = pd.read_csv(os.path.join(filepath,"all_regions_input_data_ed_daily.csv"),index_col=[0])

min_date = pd.Timestamp(2023, 1, 1)
max_date = pd.Timestamp(2025, 2, 1)

wwtps = ['Annacis', 'Iona', 'Lionsgate', 'Lulu']

metric = "MAE"

df_raw['surveillance_date'] = pd.to_datetime(df_raw['surveillance_date'])
df_raw = df_raw[(df_raw['surveillance_date']>= min_date) & (df_raw['surveillance_date']< max_date) ]
#df.set_index('region',inplace=True)
df = df_raw[df_raw['wwtp'].isin(wwtps)]
population = pop[pop['wwtp'].isin(wwtps)]['population'].values

df_pop = df.merge(pop,on = "wwtp")
df_pop["tests_per_capita"] = df_pop["total_tests_all_ages"]/df_pop["population"]
df_pop = df_pop[['surveillance_date', 'wwtp'] + label + features].rename(columns={'wwtp':
                                                                          'region'})

4. Lag features

In [2]:
# Example usage
lags_dict = {
    'lag28': 28,  # 1-week lag
    #'lag14': 14, # 2-week lag
    'lag21': 21, # 3-week lag
    #'lag3': 3    # 3-day lag
}

df_lagged = create_lagged_features_by_region(df_pop, target_column = label+features, lags_dict=lags_dict)
#print(df_lagged.head(3))

In [4]:
df_lagged.columns

Index(['surveillance_date', 'region', 'total_ed_visits',
       'total_ed_visits_lag28', 'total_ed_visits_lag21',
       'total_cases_covid_lag28', 'total_cases_covid_lag21',
       'load_trillion_covid_lag28', 'load_trillion_covid_lag21',
       'total_cases_flua_lag28', 'total_cases_flua_lag21',
       'load_trillion_flua_lag28', 'load_trillion_flua_lag21',
       'total_cases_rsv_lag28', 'total_cases_rsv_lag21',
       'load_trillion_rsv_lag28', 'load_trillion_rsv_lag21',
       'tests_per_capita_lag28', 'tests_per_capita_lag21'],
      dtype='object')

In [3]:
df_lagged.drop(columns=features, inplace=True)
df_lagged.columns

Index(['surveillance_date', 'region', 'total_ed_visits',
       'total_ed_visits_lag28', 'total_ed_visits_lag21',
       'total_cases_covid_lag28', 'total_cases_covid_lag21',
       'load_trillion_covid_lag28', 'load_trillion_covid_lag21',
       'total_cases_flua_lag28', 'total_cases_flua_lag21',
       'load_trillion_flua_lag28', 'load_trillion_flua_lag21',
       'total_cases_rsv_lag28', 'total_cases_rsv_lag21',
       'load_trillion_rsv_lag28', 'load_trillion_rsv_lag21',
       'tests_per_capita_lag28', 'tests_per_capita_lag21'],
      dtype='object')

### Fitting all data (<19 and > 19)
5. Split test and trian, and fit XGBoost using rolling window cross validation

In [5]:
df = df_lagged.sort_values(["region", "surveillance_date"])

# Assign rank *from the end* (latest date = 1)
df["rank_latest"] = df.groupby("region")["surveillance_date"] \
                      .rank(method="first", ascending=False)

# Split
test = df[df["rank_latest"] <= 21].copy()
train = df[df["rank_latest"] > 21].copy()
train.drop(columns="rank_latest", inplace=True)
test.drop(columns="rank_latest", inplace=True)

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'min_child_weight': [1, 5],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9]
}

window_size = 120
step_size= 21
best_params, region_mae_results = rolling_window_xgboost(train, region_column='region', label=label[0], features=features,
                                                         window_size=window_size, step_size=step_size, param_grid=param_grid, lags_dict=lags_dict)

print("Best parameters overall:", best_params)


Processing region: Annacis

Processing region: Iona

Processing region: Lionsgate

Processing region: Lulu
Best parameters overall: (('colsample_bytree', 0.8), ('learning_rate', 0.1), ('max_depth', 5), ('min_child_weight', 5), ('n_estimators', 100), ('subsample', 0.8))


In [6]:
final_mae_results, all_train, all_test =  train_and_predict_per_region_xgboost_bootstrapping(
    df=train,
    new_data=test,
    region_column="region",
    label=label[0],
    features=features,
    best_params=dict(best_params),
    lags_dict=lags_dict,
    n_bootstraps=100,
    block_size=21
)
 
#root_dir = Path(os.path.dirname(Path.cwd()))
#savepath = root_dir / "reports" / "figures"
#plot_train_test_predictions_bootstrapping(
#    all_train, all_test, rows=2, cols=2, savepath=savepath, start = 500, ci=0.95
#)

obs = all_test[['region', 'surveillance_date', 'Actual']].drop_duplicates().rename(columns={'surveillance_date':'time', 'region':'wwtp'})
obs_wide = (
    obs.pivot(index="time", columns="wwtp", values="Actual")
      .sort_index()
)

obs_wide = obs_wide[sorted(obs_wide.columns)]


obs_for = all_test[['bootstrap','surveillance_date', 'region', 'Pred']].rename(columns={'surveillance_date':'time', 'region':'wwtp', 'bootstrap':'sample'})
obs_for_wide = obs_for.pivot(index=['sample','time'], columns="wwtp", values = "Pred")

obs_for_wide= obs_for_wide.sort_index(level=['sample','time'])

# Sort columns alphabetically (or any fixed order you prefer)
obs_for_wide = obs_for_wide[sorted(obs_for_wide.columns)]

bootstraps = obs_for_wide.index.get_level_values("sample").unique()
dates      = obs_for_wide.index.get_level_values("time").unique()
regions    = obs_for_wide.columns


from scores.probability import crps_for_ensemble
import xarray as xr

obs_array = xr.DataArray(
    obs_wide.to_numpy(),
    dims=["time","wwtp"],
    coords={
        "time": dates,
        "wwtp": regions
    }
)
forecast_samples = xr.DataArray(
    obs_for_wide.to_numpy().reshape(len(bootstraps), len(dates), len(regions)),
    dims=["sample","time","wwtp"],
    coords={
        "sample": bootstraps,
        "time": dates,
        "wwtp": regions
    }
)

forecast_samples.to_netcdf(f"forecast_samples_xgb{suffix}.nc")
obs_array.to_netcdf(f"obs_array{suffix}.nc")

crps_matrix = crps_for_ensemble(
    obs=obs_array,                    # shape (time, wwtp)
    fcst=forecast_samples,            # shape (sample, time, wwtp)
    ensemble_member_dim="sample",     # must match forecast_samples dim
    preserve_dims=["time","wwtp"]   # must match both obs and forecast_samples
)


In [7]:
crps_mean = crps_matrix.mean(dim="time")
crps_df = crps_mean.to_dataframe(name="CRPS Xgboost")
savepath = root_dir / "reports" / "tables"
crps_df.reset_index().to_csv(os.path.join(savepath,f"crps_xgboost{suffix}.csv"), index=False)

crps_df

,CRPS Xgboost
wwtp,
Annacis,22.682588
Iona,22.392215
Lionsgate,2.088713
Lulu,4.146880


In [33]:
suffix

''

6. Use fitted model to predict using test dataset.
7. Plot test and train datasets predictions vs actuals
8. Calculate and print MAE train

In [34]:
final_mae_results, final_predictions, all_train, all_test, importance_df = train_and_predict_per_region_xgboost(
    train, test, region_column='region', label=label[0], features=features, 
    best_params=best_params,
    lags_dict = lags_dict,
    )

print("Final MAE per region:", final_mae_results)



Training final model for region: Annacis
Final MAE on training data for Annacis: 8.212060600000378

Training final model for region: Iona
Final MAE on training data for Iona: 6.658350887335477

Training final model for region: Lionsgate
Final MAE on training data for Lionsgate: 2.3262438238129137

Training final model for region: Lulu
Final MAE on training data for Lulu: 1.923075159467627
Final MAE per region: {'Annacis': 8.2121, 'Iona': 6.6584, 'Lionsgate': 2.3262, 'Lulu': 1.9231}


In [ ]:
crps_xgb = {region:None for region in wwtps}
for i in wwtps:
    current = all_test[all_test['region']==i]
    pred = current['Pred']
    observed = current['Actual']

    crps_xgb[i] = np.mean(np.abs(pred - observed))  # same as MAE


In [ ]:
crps_df = pd.DataFrame.from_dict(crps_xgb, orient='index').rename(columns={0: "CRPS Xgb"})
savepath = root_dir / "reports" / "tables"
crps_df.reset_index().to_csv(os.path.join(savepath,"crps_xgboost.csv"), index=False)
crps_xgb


In [ ]:
# coverage
lower = forecast_samples.quantile(0.025, dim="sample")
upper = forecast_samples.quantile(0.975, dim="sample")
covered = (obs_array >= lower) & (obs_array <= upper)
coverage = covered.mean().item()   # single float
coverage_per_region = covered.mean(dim="time")
coverage_per_time = covered.mean(dim="wwtp")


In [ ]:
all_train_w,  all_test_w = resampling_weekly(all_train,all_test)
plot_train_test_predictions(all_train_w, all_test_w, 2, 2, savepath = savepath)

In [ ]:
# Custom plot
plt.figure(figsize=(8, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.gca().invert_yaxis()
plt.xlabel("Gain")
plt.title("Feature Importance")
plt.show()


9. Calculate and print MAE test

In [ ]:
# Example usage:
mae_test_results = compute_mae_test(final_predictions, test, region_column='region', label=label[0])

print("MAE per region on test data:", mae_test_results)
table_mae = pd.DataFrame(list(mae_test_results.items()), columns=["Region", "MAE"])
savepath = root_dir / "reports"/ "tables"
table_mae.to_csv(os.path.join(savepath, "crps_xgb.csv"), index=False)

### Fitting only child (<19) ed visits 

In [ ]:
df_child = df_raw.copy()
df_child['child_ed_visits'] = df_child['total_ed_visits']*df_child['ed_visits_percentage_child']
df_child['child_total_cases_covid'] = df_child['total_cases_covid']*df_child['cases_percentage_child_covid']
df_child['child_total_cases_rsv'] = df_child['total_cases_rsv']*df_child['cases_percentage_child_rsv']
df_child['child_total_cases_flua'] = df_child['total_cases_flua']*df_child['cases_percentage_child_flua']


features =['child_total_cases_flua', 'child_total_cases_rsv', 'child_total_cases_covid',
           'load_trillion_flua', 'load_trillion_rsv','load_trillion_covid',
           #'cases_percentage_adult_covid', 'cases_percentage_child_covid',
           #'cases_percentage_adult_flua', 'cases_percentage_child_flua',
           #'cases_percentage_adult_rsv', 'cases_percentage_child_rsv', 
           ]
label = ['child_ed_visits']

df_child = df_child[['surveillance_date', 'wwtp'] + label + features].rename(columns={'wwtp':
                                                                           'region'})

# Example usage
lags_dict = {
    'lag7': 7,  # 1-week lag
    'lag14': 14, # 2-week lag
    'lag21': 21, # 3-week lag
    'lag3': 3    # 3-day lag
}

df_child_lagged = create_lagged_features_by_region(df_child, target_column = label+features, lags_dict=lags_dict)
print(df_child_lagged.head())


In [ ]:
test = df_child_lagged[df_child_lagged['surveillance_date'] >= pd.to_datetime("2024-12-1")]
train = df_child_lagged[df_child_lagged['surveillance_date'] < pd.to_datetime("2024-12-1")]

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'min_child_weight': [1, 5],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9]
}

window_size = 120
step_size= 60
best_params, region_mae_results = rolling_window_xgboost(train, region_column='region', label=label[0], features=features,
                                                         window_size=window_size, step_size=step_size, param_grid=param_grid, lags_dict=lags_dict)

print("Best parameters overall:", best_params)

In [ ]:
final_mae_results, final_predictions, child_train, child_test = train_and_predict_per_region_xgboost(
    train, test, region_column='region', label=label[0], features=features, 
    best_params=best_params, lags_dict= lags_dict)

print("Final MAE per region:", final_mae_results)


In [ ]:
# Example usage:
mae_test_results = compute_mae_test(final_predictions, test, region_column='region', label=label[0])

print("MAE per region on test data:", mae_test_results)


In [ ]:
plot_train_test_predictions(child_train, child_test)

In [ ]:
all_train_w,  all_test_w = resampling_weekly(all_train,all_test)
plot_train_test_predictions(all_train_w, all_test_w)

### Bayesian Optimization

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

def bayesopt_rolling_window_xgboost(df, region_column, label, features, window_size, step_size, lags_dict, n_trials=30):
    unique_regions = df[region_column].unique()

    def objective(trial):
        # Sample hyperparameters
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'objective': 'reg:absoluteerror',
            'n_jobs': -1
        }

        mae_scores = []

        for region in unique_regions:
            df_region = df[df[region_column] == region].copy()

            for start in range(0, len(df_region) - window_size - step_size, step_size):
                train_data = df_region.iloc[start:start + window_size]
                val_data = df_region.iloc[start + window_size:start + window_size + step_size]

                X_train = train_data[features + [f'{label}_{lag_name}' for lag_name in lags_dict.keys()]]
                y_train = train_data[label]
                X_val = val_data[features + [f'{label}_{lag_name}' for lag_name in lags_dict.keys()]]
                y_val = val_data[label]

                model = XGBRegressor(**params)
                model.fit(X_train, y_train)

                y_pred = model.predict(X_val)
                mae = mean_absolute_error(y_val, y_pred)
                mae_scores.append(mae)

        return np.mean(mae_scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    print("Best parameters:", study.best_params)
    return study.best_params, study


In [ ]:
best_params, study = bayesopt_rolling_window_xgboost(
    train,
    region_column='region',
    label=label[0],
    features=features,
    window_size=120,
    step_size=60,
    lags_dict=lags_dict,
    n_trials=50  # adjust for more thorough search
)


In [ ]:
best_params

In [ ]:
final_mae_results, final_predictions, all_train, all_test, importance_df = train_and_predict_per_region_xgboost(
    train, test, region_column='region', label=label[0], features=features, 
    best_params=best_params,
    lags_dict = lags_dict,
    )

print("Final MAE per region:", final_mae_results)


In [ ]:
crps_df = pd.DataFrame.from_dict(crps_xgb, orient='index').rename(columns={0: "CRPS Xgb"})
#savepath = root_dir / "reports" / "tables"
#crps_df.reset_index().to_csv(os.path.join(savepath,"crps_xgboost.csv"), index=False)
crps_xgb


In [ ]:
root_dir = Path(os.path.dirname(Path.cwd()))
savepath = root_dir / "reports" / "figures"
plot_train_test_predictions(all_train, all_test, rows = 2, cols = 2, savepath = savepath)